# Lab 06: Regularization and Feature Engineering

In this lab, we will explore two important techniques in machine learning: regularization and feature engineering.
We will use the California housing dataset, which contains information about various features of houses in California and their corresponding prices.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

plt.style.use("bmh")
plt.rcParams["font.family"] = ["Helvetica"]
np.random.seed(30009000)

data = fetch_california_housing(as_frame=True)
data.frame

## Feature Engineering

Let's start by exploring the dataset values.

In [ ]:
data.frame.describe()

As we can see, the features have different scales and ranges.
To make learning easier for our models, we can apply some transformations to the features.

But first let's do the usual train-validation-test split of the dataset.

In [ ]:
X, y = data.data, data.target

# 60-20-20 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

print(
  f"Training set size: {X_train.shape[0]}, Validation set size: {X_val.shape[0]}, Test set size: {X_test.shape[0]}"
)

First we make sure the values of the features are on a similar scale by applying standardization.
This means we will transform the features to have zero mean and unit variance by applying:

$$
  x' = \frac{x - \mu}{\sigma} ,
$$

where $\mu$ is the mean of the feature and $\sigma$ is the standard deviation of the feature.
This is done separately for each column of the dataset.

The following code will compute the mean and standard deviation of the training set and apply the transformation to the training, validation, and test sets.
Note that we fit the scaler only on the training set to avoid data leakage from the validation and test sets.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)  # Compute the mean and std of the training set

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

Note that it is often useful to normalize/scale the target variable as well, especially if it has a wide range of values.

## Regularization

Regularization is a technique used to prevent overfitting by adding a penalty term to the loss function.
Usually, this is done by constraining the values of the parameters of the model.
For example, the L2 regularization (Ridge regression) adds a penalty term proportional to the square of the magnitude of the coefficients, while L1 regularization (Lasso regression) adds a penalty term proportional to the absolute value of the coefficients.


In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_error

# Alpha is the regularization strength. In the slides, it's lambda.
models = [
  Ridge(alpha=0.1),
  Lasso(alpha=0.1)
]

for model in models:
  model.fit(X_train_scaled, y_train)
  mae_val = mean_absolute_error(y_val, model.predict(X_val_scaled))
  print(f"{model.__class__.__name__}")
  print(f"Coefficients: {model.coef_}")
  print(f"Validation MAE: {mae_val:.4f}")
  print()

Let's see the effect of regularization on the coefficients of a linear regression model.

In [ ]:
import ipywidgets as widgets
from IPython.display import display


def run_sweep(alpha_exp, model_type):
  alpha = 10 ** alpha_exp

  model = Ridge(alpha=alpha) if model_type == "Ridge" else Lasso(alpha=alpha, max_iter=10000)
  model.fit(X_train_scaled, y_train)

  mae_train = mean_absolute_error(y_train, model.predict(X_train_scaled))
  mae_val = mean_absolute_error(y_val,   model.predict(X_val_scaled))
  n_zero = np.sum(np.abs(model.coef_) < 1e-4)

  # --- Left: coefficient bar chart ---
  colors = ["steelblue" if abs(c) >= 1e-4 else "lightgray" for c in model.coef_]
  plt.bar(np.arange(X_train_scaled.shape[1]), model.coef_, color=colors)
  plt.axhline(0, color="black", linewidth=0.8)
  plt.title(rf"$\alpha: $ {alpha:.0e}   |   Train MAE: {mae_train:.4f}   |   Val MAE: {mae_val:.4f}   |   norm = {np.linalg.norm(model.coef_, ord=1 if model_type == 'Lasso' else 2):.4f}   |   #Zero Coefficients: {n_zero}")
  plt.xlabel("Feature")
  plt.ylabel("Coefficient")

  plt.show()


widgets.interact(
  run_sweep,
  alpha_exp=widgets.FloatSlider(
    value=0, min=-3, max=3, step=0.1,
    description=r"$\log \alpha$",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
  ),
  model_type=widgets.RadioButtons(
    options=["Ridge", "Lasso"],
    description="Model:",
    style={"description_width": "initial"}
  )
);

## Selecting the Regularization Strength

Just like last week, we can use K-fold cross-validation to select the best value of the regularization strength $\alpha$.
We will use Ridge regression for this example, but the same process can be applied to Lasso regression as well.

In [ ]:
from sklearn.model_selection import cross_val_score

# Between log(-3) and log(3)
alphas = np.logspace(-3, 3, 10)

cross_val_maes = []

for alpha in alphas:
  model = Ridge(alpha=alpha)
  scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="neg_mean_absolute_error")
  cross_val_maes.append(-scores.mean())

best_alpha = alphas[np.argmin(cross_val_maes)]
print(f"Best alpha = {best_alpha}")

Then we train the final model with the best alpha and evaluate it on the test set.

In [ ]:
final_model = Ridge(alpha=best_alpha)
final_model.fit(X_train_scaled, y_train)

test_mae = mean_absolute_error(y_test, final_model.predict(X_test_scaled))
print(f"Test MAE: {test_mae:.4f}")